# Step 4: Model Evaluation & Performance Review

This notebook evaluates **base (pretrained)** and **fine-tuned (LoRA)** models on the full held-out test split (`data/finetuning_dataset_test.jsonl`, 20% from `dataset_builder.py`). Ground truth is the Gemini labels in each example's `response` field.

[`pipeline/evaluator.py`](pipeline/evaluator.py) saves per-sample checkpoints under `data/evaluation_checkpoints/` so long runs can resume, and writes `data/evaluation_results.csv` with a `Variant` column (`base` vs `fine-tuned`).

## 0. Setup Google Colab (Mount Drive)
Run this block to mount your Google Drive and enter the project directory.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    %cd "/content/drive/MyDrive/04 - University/04.02 - Adelaide University/04.02.02 - Sem 1 2026/02 - Deep Learning Applications/0 - Final Project/call-contextual-extractor"
except ImportError:
    print("Not running in Google Colab, skipping drive mount.")

In [ ]:
try:
    import google.colab
    !pip install -q condacolab
    import condacolab
    condacolab.install()
except ImportError:
    print("Not running in Colab. Skipping Conda setup.")

In [ ]:
try:
    # from google.colab import drive
    # drive.mount("/content/drive")
    # %cd "/content/drive/MyDrive/04 - University/04.02 - Adelaide University/04.02.02 - Sem 1 2026/02 - Deep Learning Applications/0 - Final Project/call-contextual-extractor"

    print("\n--- Installing Environment ---")
    !conda env update -n base -f environment.yml

    print("\n--- Installing Colab Unsloth Drivers ---")
    !pip install pandas
    !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
    !pip install --no-deps "xformers<0.0.27" peft accelerate bitsandbytes
except ImportError:
    print("Not running in Colab. Skipping mount and environment update.")

## 1. Run Evaluator Pipeline

Runs the **full test set** for each model below:

| Backbone | Base (Hugging Face) | Fine-tuned LoRA (Colab/CUDA) | Mac MLX eval |
|----------|---------------------|------------------------------|--------------|
| Qwen3.5-2B | `Qwen/Qwen3.5-2B` | `data/models/Qwen3.5-2B_lora` | `data/models/Qwen3.5-2B_merged` |
| Qwen3.5-0.8B | `Qwen/Qwen3.5-0.8B` | `data/models/Qwen3.5-0.8B_lora` | `data/models/Qwen3.5-0.8B_merged` |
| Gemma-4 E4B | `google/gemma-4-E4B` | `data/models/gemma-4-E4B_lora` | `data/models/gemma-4-E4B_merged` |

**Mac note:** LoRA folders from Colab are **PEFT format** (`adapter_model.safetensors`). MLX cannot load them directly (`num_layers` error). After training, run once on Colab:

`python pipeline/export_merged.py --lora data/models/Qwen3.5-2B_lora`

Sync the `*_merged` folders to your Mac. The evaluator still lists `*_lora` paths; it auto-uses `*_merged` when present.

Metrics tracked:
1. JSON parsability (`Valid_JSON_%`)
2. Boolean accuracy (busy)
3. Soft string match (sector, scheduled time)
4. MAE / MSE / ±1 accuracy (interested, rating)

Progress is printed per sample with **running** JSON/Busy/Sector scores. While the run is active you can open:

- `data/evaluation_results_live.csv` — aggregate metrics so far (updates every sample)
- `data/evaluation_progress.json` — current model, sample index, running metrics
- `data/evaluation_checkpoints/*.samples.jsonl` — per-example prediction vs ground truth

Interrupted runs resume when you re-run the cell (omit `--no-resume`).

In [1]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Colab default path (also works after the mount/%cd setup cells above)
COLAB_PROJECT = Path(
    "/content/drive/MyDrive/04 - University/04.02 - Adelaide University/"
    "04.02.02 - Sem 1 2026/02 - Deep Learning Applications/0 - Final Project/"
    "call-contextual-extractor"
)


def project_root() -> Path:
    for candidate in (Path.cwd(), COLAB_PROJECT):
        if (candidate / "pipeline" / "evaluator.py").exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Project root not found (missing pipeline/evaluator.py). "
        "Run the Colab mount / %cd cells above, or cd into the repo."
    )


def _running_in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def evaluation_python() -> str:
    """Use this notebook's kernel locally; on Colab prefer python3.11 (Unsloth pip target)."""
    if not _running_in_colab():
        return sys.executable
    for name in ("python3.11", "python3", "python"):
        path = shutil.which(name)
        if path:
            return path
    return sys.executable


ROOT = project_root()
os.chdir(ROOT)

BASE_MODELS = [
    "Qwen/Qwen3.5-2B",
    "Qwen/Qwen3.5-0.8B",
]
LORA_MODELS = [
    "data/models/Qwen3.5-2B_lora",
    "data/models/Qwen3.5-0.8B_lora",
]

lora_available = [p for p in LORA_MODELS if (ROOT / p).exists()]
all_models = BASE_MODELS + lora_available
py = evaluation_python()

print("Project root:", ROOT)
print("Python executable:", py)
print("Base models (always evaluated):", BASE_MODELS)
print("Fine-tuned adapters found:", lora_available or "(none — train in 03_FineTuning.ipynb first)")
print(f"Total models this run: {len(all_models)}")
print("Test split: data/finetuning_dataset_test.jsonl")
print("Checkpoints: data/evaluation_checkpoints/ (resume on re-run)")
print("Live CSV: data/evaluation_results_live.csv (open while running)\n")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if _running_in_colab():
    # Colab: kernel Python often lacks Unsloth; use the env where pip installed it.
    print("Colab: launching evaluator subprocess...")
    subprocess.run(
        [py, "pipeline/evaluator.py", "--models", *all_models],
        cwd=ROOT,
        check=True,
    )
else:
    # Local (M4 etc.): run in this kernel — same env, no duplicate model loads across processes.
    print("Local: running evaluate_models() in this notebook kernel...")
    import importlib
    import pipeline.evaluator as evaluator_module

    importlib.reload(evaluator_module)
    evaluator_module.evaluate_models(all_models)

Project root: /Users/megatunger/Github/call-contextual-extractor
Python executable: /opt/anaconda3/bin/python
Base models (always evaluated): ['Qwen/Qwen3.5-2B', 'Qwen/Qwen3.5-0.8B']
Fine-tuned adapters found: ['data/models/Qwen3.5-2B_lora', 'data/models/Qwen3.5-0.8B_lora']
Total models this run: 4
Test split: data/finetuning_dataset_test.jsonl
Checkpoints: data/evaluation_checkpoints/ (resume on re-run)
Live CSV: data/evaluation_results_live.csv (open while running)

Local: running evaluate_models() in this notebook kernel...
Loading test dataset...
Evaluating on 110 held-out test examples from /Users/megatunger/Github/call-contextual-extractor/data/finetuning_dataset_test.jsonl.
Checkpoints: data/evaluation_checkpoints (use --no-resume to start fresh)
Live summary: data/evaluation_results_live.csv
Per-sample logs: data/evaluation_checkpoints/*.samples.jsonl
Progress file: data/evaluation_progress.json

Evaluating model: Qwen/Qwen3.5-2B


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Unsloth: Loading Qwen/Qwen3.5-2B via mlx-vlm (VLM, runtime 4-bit affine quantization)...


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

[INFO] Quantized model with 8.867 bits per weight.
Unsloth: text_only=True requested for a multimodal wrapper; keeping the model on the mlx-vlm path and returning its tokenizer.
Qwen3.5-2B (base): 1/110 (0.9%) | JSON 0.0% Busy 0% Sector 0% | elapsed 0s | ETA 0s


...

Qwen3.5-0.8B (fine-tuned): 50/110 (45.5%) | JSON 42.73% Busy 51.06% Sector 46.81% | elapsed 83s | ETA 100s
Qwen3.5-0.8B (fine-tuned): 51/110 (46.4%) | JSON 43.64% Busy 52.08% Sector 45.83% | elapsed 86s | ETA 99s
Qwen3.5-0.8B (fine-tuned): 52/110 (47.3%) | JSON 44.55% Busy 51.02% Sector 44.9% | elapsed 87s | ETA 97s
Qwen3.5-0.8B (fine-tuned): 53/110 (48.2%) | JSON 45.45% Busy 50.0% Sector 46.0% | elapsed 91s | ETA 98s


Prefill: 100%|████████████████████████████████████████████▉| 5950/5951 [00:04<00:00, 1220.94tok/s]


Qwen3.5-0.8B (fine-tuned): 54/110 (49.1%) | JSON 46.36% Busy 49.02% Sector 45.1% | elapsed 93s | ETA 96s
Qwen3.5-0.8B (fine-tuned): 55/110 (50.0%) | JSON 47.27% Busy 48.08% Sector 44.23% | elapsed 99s | ETA 99s
Qwen3.5-0.8B (fine-tuned): 56/110 (50.9%) | JSON 48.18% Busy 49.06% Sector 43.4% | elapsed 100s | ETA 96s
Qwen3.5-0.8B (fine-tuned): 57/110 (51.8%) | JSON 49.09% Busy 50.0% Sector 44.44% | elapsed 101s | ETA 94s
Qwen3.5-0.8B (fine-tuned): 58/110 (52.7%) | JSON 50.0% Busy 50.91% Sector 45.45% | elapsed 103s | ETA 92s
Qwen3.5-0.8B (fine-tuned): 59/110 (53.6%) | JSON 50.91% Busy 51.79% Sector 46.43% | elapsed 104s | ETA 90s
Qwen3.5-0.8B (fine-tuned): 60/110 (54.5%) | JSON 51.82% Busy 52.63% Sector 45.61% | elapsed 105s | ETA 88s
Qwen3.5-0.8B (fine-tuned): 61/110 (55.5%) | JSON 52.73% Busy 51.72% Sector 44.83% | elapsed 107s | ETA 86s
Qwen3.5-0.8B (fine-tuned): 62/110 (56.4%) | JSON 53.64% Busy 50.85% Sector 44.07% | elapsed 111s | ETA 86s
Qwen3.5-0.8B (fine-tuned): 63/110 (57.3%) |

Prefill: 100%|█████████████████████████████████████████████▉| 2325/2326 [00:02<00:00, 998.77tok/s]


Qwen3.5-0.8B (fine-tuned): 81/110 (73.6%) | JSON 69.09% Busy 52.63% Sector 43.42% | elapsed 141s | ETA 50s
Qwen3.5-0.8B (fine-tuned): 82/110 (74.5%) | JSON 70.0% Busy 51.95% Sector 42.86% | elapsed 146s | ETA 50s
Qwen3.5-0.8B (fine-tuned): 83/110 (75.5%) | JSON 70.91% Busy 52.56% Sector 42.31% | elapsed 149s | ETA 48s
Qwen3.5-0.8B (fine-tuned): 84/110 (76.4%) | JSON 71.82% Busy 53.16% Sector 43.04% | elapsed 150s | ETA 46s
Qwen3.5-0.8B (fine-tuned): 85/110 (77.3%) | JSON 72.73% Busy 53.75% Sector 43.75% | elapsed 151s | ETA 45s
Qwen3.5-0.8B (fine-tuned): 86/110 (78.2%) | JSON 73.64% Busy 54.32% Sector 43.21% | elapsed 152s | ETA 43s
Qwen3.5-0.8B (fine-tuned): 87/110 (79.1%) | JSON 74.55% Busy 54.88% Sector 42.68% | elapsed 155s | ETA 41s
Qwen3.5-0.8B (fine-tuned): 88/110 (80.0%) | JSON 75.45% Busy 54.22% Sector 42.17% | elapsed 156s | ETA 39s
Qwen3.5-0.8B (fine-tuned): 89/110 (80.9%) | JSON 76.36% Busy 54.76% Sector 42.86% | elapsed 158s | ETA 37s
Qwen3.5-0.8B (fine-tuned): 90/110 (81.

## 2. Full Results & Visualizations

Loads `data/evaluation_results.csv` (all base + fine-tuned rows), prints the full metrics table, a base-vs-fine-tuned comparison per backbone, and bar charts grouped by model with `Variant` hue.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

RESULTS_PATH = "data/evaluation_results.csv"
LIVE_RESULTS_PATH = "data/evaluation_results_live.csv"
PROGRESS_PATH = "data/evaluation_progress.json"
METRIC_COLS = [
    "Valid_JSON_%",
    "Busy_Accuracy_%",
    "Sector_Match_%",
    "Schedule_Match_%",
    "Interested_MAE",
    "Interested_MSE",
    "Interested_Acc_±1_%",
    "Rating_MAE",
    "Rating_MSE",
    "Rating_Acc_±1_%",
]
VARIANT_ORDER = {"base": 0, "fine-tuned": 1}

try:
    results_file = LIVE_RESULTS_PATH if Path(LIVE_RESULTS_PATH).exists() else RESULTS_PATH
    if results_file == LIVE_RESULTS_PATH:
        print(f"(Using live results: {LIVE_RESULTS_PATH})")
    df = pd.read_csv(results_file)
    if "Variant" not in df.columns:
        df["Variant"] = "unknown"

    df = df.sort_values(["Model", df["Variant"].map(VARIANT_ORDER).fillna(99)])
    df["Label"] = df["Model"] + " (" + df["Variant"] + ")"

    if Path(PROGRESS_PATH).exists():
        progress = json.loads(Path(PROGRESS_PATH).read_text(encoding="utf-8"))
        print("=== Run status ===")
        print(
            f"Last update: {progress.get('updated_at')} | "
            f"current: {progress.get('current_model')} "
            f"({progress.get('samples_done', '?')}/{progress.get('num_test', '?')})"
        )

    print("\n=== Full evaluation results ===")
    display(df)

    if df["Variant"].nunique() > 1:
        print("\n=== Base vs fine-tuned (per backbone) ===")
        compare = df.pivot(index="Model", columns="Variant", values=METRIC_COLS)
        display(compare)

    sns.set_theme(style="whitegrid")
    fig, axes = plt.subplots(2, 2, figsize=(20, 11))
    axes = axes.flatten()

    sns.barplot(data=df, x="Model", y="Valid_JSON_%", hue="Variant", ax=axes[0])
    axes[0].set_title("Valid JSON Output Rate (%)")
    axes[0].set_ylim(0, 100)
    axes[0].tick_params(axis="x", rotation=15)

    df_melted = df.melt(
        id_vars=["Model", "Variant", "Label"],
        value_vars=[
            "Busy_Accuracy_%",
            "Sector_Match_%",
            "Schedule_Match_%",
            "Interested_Acc_±1_%",
            "Rating_Acc_±1_%",
        ],
        var_name="Metric",
        value_name="Score",
    )
    sns.barplot(data=df_melted, x="Label", y="Score", hue="Metric", ax=axes[1])
    axes[1].set_title("Field Exact & Tolerance Accuracy (%)")
    axes[1].set_ylim(0, 100)
    axes[1].tick_params(axis="x", rotation=25)
    axes[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)

    df_mae = df.melt(
        id_vars=["Model", "Variant", "Label"],
        value_vars=["Interested_MAE", "Rating_MAE"],
        var_name="Metric",
        value_name="MAE",
    )
    sns.barplot(data=df_mae, x="Label", y="MAE", hue="Metric", ax=axes[2])
    axes[2].set_title("Rating Error (MAE — lower is better)")
    axes[2].tick_params(axis="x", rotation=25)

    df_mse = df.melt(
        id_vars=["Model", "Variant", "Label"],
        value_vars=["Interested_MSE", "Rating_MSE"],
        var_name="Metric",
        value_name="MSE",
    )
    sns.barplot(data=df_mse, x="Label", y="MSE", hue="Metric", ax=axes[3])
    axes[3].set_title("Rating Error (MSE — lower is better)")
    axes[3].tick_params(axis="x", rotation=25)

    plt.tight_layout()
    plt.show()

    print(f"\n{len(df)} model runs loaded from {RESULTS_PATH}")

except FileNotFoundError:
    print(f"No results yet. Run the evaluator cell, or check {PROGRESS_PATH} while it runs.")
